# Stage 0 · CPU notebook

Source readiness, verified spectral cache and explicit projection reporting. No model inference or GPU runtime is required. The GPU notebook shares the saved plans.


Ticket 10: set `CONDITIONAL_EXTENSION = True` in section 1 only after CPU sections 26–28 produce an applicable catalogue. The same qualification/collection/review cells then use separately named extension products. Keep the same cumulative Stage 0 usage ledger; the ceiling remains 20 units. Build the catalogue on CPU first, use the GPU notebook for the separately reviewed qualification/collection, then return here with the same flag for sections 17–22.


## 1. Storage and checkout
Mount Drive when in Colab; select the existing Stage 0 release checkout. CPU runtime is sufficient for all current cells.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

RELEASE_COMMIT = 'd65e52d11d7e8aee737b33e33baa72033470cca6'
REPOSITORY_URL = 'https://github.com/mahadikprasad15/spar-pilot.git'
# Only the dedicated Colab checkout is pinned; local work is never switched.
REPO_DIR = Path('/content/spar-stage0') if Path('/content').is_dir() else Path.cwd()
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts') if Path('/content').is_dir() else REPO_DIR / 'artifacts'
STAGE0_NAME = 'stage0-geometry-v1'
CONDITIONAL_EXTENSION = False  # True selects the separately named Ticket 10 workflow.
USE_BATCH16 = not CONDITIONAL_EXTENSION  # Approved primary schedule: behavior16, internals1.
BATCH_NAME = STAGE0_NAME + '-behavior-b16-v2'  # v1 failed the superseded 3E batch gate; evidence preserved (ADR 0014).
PLAN = ARTIFACT_ROOT / 'plans' / STAGE0_NAME / 'stage0.plan.json'

if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
if Path('/content').is_dir():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPO_DIR)], check=True)
    changed = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True)
    if changed.strip():
        raise RuntimeError('The dedicated checkout has local changes. Preserve them before selecting a release.')
    subprocess.run(['git', 'fetch', 'origin', RELEASE_COMMIT], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', RELEASE_COMMIT], cwd=REPO_DIR, check=True)
if not (REPO_DIR / 'pilot_eval').is_dir():
    raise RuntimeError('Select the repository checkout before running the workflow.')

def run_stage0(operation, *arguments):
    """Stream the command's output; on failure, raise with its last lines so the real error is visible."""
    command = [sys.executable, '-u', '-m', 'pilot_eval', 'stage0-' + operation,
               '--output-root', str(ARTIFACT_ROOT), *map(str, arguments)]
    process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    try:
        for line in process.stdout:
            print(line, end='')
            tail = (tail + [line])[-30:]
        process.wait()
    except BaseException:
        # Stop the worker too: Ctrl-C first so it records timing and releases its writer,
        # then terminate, then kill; always reap it so no GPU job outlives the cell.
        import signal
        process.send_signal(signal.SIGINT)
        for escalate, timeout in [(process.terminate, 60), (process.kill, 30), (None, None)]:
            try:
                process.wait(timeout=timeout)
                break
            except subprocess.TimeoutExpired:
                escalate()
        raise
    if process.returncode:
        raise RuntimeError(f'stage0-{operation} failed (exit {process.returncode}):\n' + ''.join(tail))
    return subprocess.CompletedProcess(command, 0)


## 2. Select and prepare
Bind the SFT/GRPO prepared sources. Optional inputs remain distinguishable from verified endpoint adapters.

In [ ]:
# Select existing prepared products; paths are relative to ARTIFACT_ROOT.
QUALIFICATION_NAME = STAGE0_NAME + ('-intruder-qualification-v1' if CONDITIONAL_EXTENSION else '-loader-qualification-v1')
SELECTION = ARTIFACT_ROOT / 'notebook' / QUALIFICATION_NAME / 'source-selection.json'
settings = {
    'sources': {
        'sft': 'plans/pilot3-inputs-v1/activation.prepared.json',
        'grpo': 'plans/pilot4-grpo-fp32-v1-writes-v1/activation.prepared.json',
        'random': 'plans/pilot4-grpo-fp32-v1-random-writes-v1/activation.prepared.json',
    },
    # Optional product paths must point to explicit file-hash completion receipts.
    'products': {},
}
SELECTION.parent.mkdir(parents=True, exist_ok=True)
if SELECTION.exists() and json.loads(SELECTION.read_text()) != settings:
    raise RuntimeError('Saved source selection differs. Inspect it before selecting another plan name.')
SELECTION.write_text(json.dumps(settings, indent=2) + '\n')
run_stage0('prepare', '--selection', SELECTION, '--name', STAGE0_NAME)


## 3. Audit or resume
Verify selected config, seal and adapter payloads; compare actual zero-initialization tensors. Sealed source units are reused. No optimizer-state replay, model download or inference.

In [ ]:
run_stage0('audit', '--config', PLAN)


## 4. Restore saved status
Safe in a fresh kernel after section 1. Reports audit evidence, missing prerequisites and the cumulative 20-unit budget. Unknown cost is not zero cost.

In [ ]:
run_stage0('status', '--config', PLAN)


## 5. Prepare the local spectral cache
Input: completed source audit and pinned local safetensor files. Output: an immutable cache plan. This verifies the same FP32 base hash recorded by training; it neither downloads nor constructs a model. Stage weights using the guide before this cell.

In [ ]:
AUDIT_RESULT = ARTIFACT_ROOT / json.loads(PLAN.read_text())['run_path'] / 'results' / 'source-audit.json'
audit = json.loads(AUDIT_RESULT.read_text())
revision = audit['sources']['sft']['identity']['model_revision']
WEIGHTS_DIR = ARTIFACT_ROOT / 'inputs' / 'models' / revision
SPECTRAL_NAME = STAGE0_NAME + '-spectral-v1'
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / SPECTRAL_NAME / 'stage0.spectral.json'
if not (WEIGHTS_DIR / 'config.json').is_file():
    raise RuntimeError('Stage the pinned local weight snapshot at ' + str(WEIGHTS_DIR) + '; see docs/stage0-geometry.md.')
run_stage0('spectral-prepare', '--audit', PLAN, '--weights-dir', WEIGHTS_DIR, '--name', SPECTRAL_NAME)


## 6. Measure one cache unit
Output: one sealed module decomposition, measured seconds and retained bytes. This is a bounded first measurement, not a guaranteed full-run ETA. Rerunning reuses the sealed unit.

In [ ]:
run_stage0('spectral-run', '--config', SPECTRAL_PLAN, '--max-new-units', 1)


## 7. Complete or resume the cache
Decompose only missing modules. Each sealed unit retains all singular values and enough bases for k30/k50/k70 and the top-m intruder rule. Source identities, reconstruction and orthogonality are checked. No trained overlap numbers are revealed.

In [ ]:
run_stage0('spectral-run', '--config', SPECTRAL_PLAN)


## 8. Verify saved cache
Safe after restart once section 1 is run and SPECTRAL_PLAN is restored. Output: verified cache summary; later edit stages can consume these bases without opening the overlap report.

In [ ]:
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-spectral-v1') / 'stage0.spectral.json'
run_stage0('spectral-verify', '--config', SPECTRAL_PLAN)


## 9. Optional trained-overlap reveal
Leave disabled if preserving the conditional gradient-to-overlap prediction. To reveal, supply an explicit review recording either frozen-before-reveal predictions or exploratory status; the guide documents its fields. Outputs: per-module JSONL/CSV, plot and completion receipt.

In [ ]:
REVEAL_TRAINED_OVERLAPS = False
OVERLAP_REVIEW = ARTIFACT_ROOT / 'notebook' / QUALIFICATION_NAME / 'overlap-review.json'
if REVEAL_TRAINED_OVERLAPS:
    run_stage0('spectral-report', '--config', SPECTRAL_PLAN, '--review', OVERLAP_REVIEW,
               '--name', STAGE0_NAME + '-principal-overlap-v1')
else:
    print('Trained overlaps remain unrevealed. Cache is ready for edit construction.')


## 10. Prepare the principal intervention catalogue
Input: verified spectral cache and audited endpoint adapters. Output: immutable edit plan. This uses the cache without opening the trained-overlap report.

In [ ]:
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-spectral-v1') / 'stage0.spectral.json'
EDIT_NAME = STAGE0_NAME + '-principal-edits-v1'
EDIT_PLAN = ARTIFACT_ROOT / 'plans' / EDIT_NAME / 'stage0.edits.json'
run_stage0('edits-prepare', '--spectral', SPECTRAL_PLAN, '--name', EDIT_NAME)


## 11. Build or resume adapters
Output: ten separate PEFT adapters, per-module check records and saved random bases. Original source adapters are never modified. Completed module units are reused after interruption. CPU only; no model is loaded.

In [ ]:
run_stage0('edits-run', '--config', EDIT_PLAN)


## 12. Verify the catalogue
Safe after restart following section 1. Output: paths, operations and verified payload identities. Construction completion does not mean GPU qualification or evaluation has passed.

In [ ]:
EDIT_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-principal-edits-v1') / 'stage0.edits.json'
run_stage0('edits-verify', '--config', EDIT_PLAN)


## 13. Restore paired collection status (CPU)
After section 1, restore the exact Ticket 05 plan produced by the GPU notebook. This checks saved batches and budget without importing a model runtime. An incomplete run stays incomplete; this cell does not regenerate outputs.


In [ ]:
QUALIFICATION_NAME = STAGE0_NAME + ('-intruder-qualification-v1' if CONDITIONAL_EXTENSION else '-loader-qualification-v1')
EVALUATION_PLAN = ARTIFACT_ROOT / 'plans' / (BATCH_NAME if USE_BATCH16 else QUALIFICATION_NAME) / 'stage0.evaluation.json'
assert EVALUATION_PLAN.is_file(), 'Prepare paired collection in GPU notebook section 12 first.'
run_stage0('evaluation-status', '--config', EVALUATION_PLAN)
EVALUATION_CONFIG = json.loads(EVALUATION_PLAN.read_text())
EVALUATION_DIR = ARTIFACT_ROOT / EVALUATION_CONFIG['run_path']


## 14. Verify and inspect raw behavioral results (CPU)
Requires completed GPU collection. Verification reconciles all 1,500 responses, token/cap metadata, scorer outputs and descriptive counts against the sealed batches. The summary contains automatic scores; paired statistics and targeted answer review are later Ticket 07 work.


In [ ]:
run_stage0('evaluation-verify', '--config', EVALUATION_PLAN)
from IPython.display import Markdown, display
display(Markdown((EVALUATION_DIR / 'results' / 'results.md').read_text()))
print('Raw responses:', EVALUATION_DIR / 'results' / 'responses.jsonl')
print('Structured summary:', EVALUATION_DIR / 'results' / 'results.json')
print('Table:', EVALUATION_DIR / 'results' / 'summary.csv')


## 15. Restore fixed-context collection status (CPU)
The GPU notebook prepares and collects the KL/write product. This reads its sealed progress without running inference. Do not recover a writer while the GPU notebook is still active.


In [ ]:
DRIFT_NAME = STAGE0_NAME + ('-intruder-fixed-context-v1' if CONDITIONAL_EXTENSION else ('-batch16-fp64-fixed-context-v3' if USE_BATCH16 else '-fixed-context-v1'))
DRIFT_PLAN = ARTIFACT_ROOT / 'plans' / DRIFT_NAME / 'stage0.drift.json'
assert DRIFT_PLAN.is_file(), 'Prepare fixed-context collection in GPU notebook section 16 first.'
DRIFT_CONFIG = json.loads(DRIFT_PLAN.read_text())
DRIFT_DIR = ARTIFACT_ROOT / DRIFT_CONFIG['run_path']
run_stage0('drift-status', '--config', DRIFT_PLAN)


## 16. Verify fixed-context KL and effective-write results (CPU)
Rebuilds summaries from sealed per-position KL and per-document/view/module write sums. Counted target positions are shifted for KL; write positions are unshifted. The next ticket provides paired intervals, review overlays and combined behavioral interpretation.


In [ ]:
run_stage0('drift-verify', '--config', DRIFT_PLAN)
result = json.loads((DRIFT_DIR / 'results' / 'results.json').read_text())
print('Measurement complete:', result['measurement_complete'])
print('Execution kind:', result['execution_kind'])
print('Corpus/view cells:', len(result['cells']))
for cell in result['cells']:
    print(cell['variant'], cell['corpus'], cell['view'], cell['kl'])
print('Full results:', DRIFT_DIR / 'results' / 'results.json')
print('Document sums:', DRIFT_DIR / 'results' / 'per-document.jsonl')


## 17. Restore the primary analysis plan (CPU)
Use the saved plan from GPU section 12A. If it is absent, preparation is allowed now but records that collection already started; it does not claim preregistration. This restores paths without inference.


In [ ]:
QUALIFICATION_NAME = STAGE0_NAME + ('-intruder-qualification-v1' if CONDITIONAL_EXTENSION else '-loader-qualification-v1')
EXECUTION_PLAN = ARTIFACT_ROOT / 'plans' / (BATCH_NAME if USE_BATCH16 else QUALIFICATION_NAME) / 'stage0.execution.json'
EVALUATION_PLAN = EXECUTION_PLAN.parent / 'stage0.evaluation.json'
DRIFT_NAME = STAGE0_NAME + ('-intruder-fixed-context-v1' if CONDITIONAL_EXTENSION else ('-batch16-fp64-fixed-context-v3' if USE_BATCH16 else '-fixed-context-v1'))
DRIFT_PLAN = ARTIFACT_ROOT / 'plans' / DRIFT_NAME / 'stage0.drift.json'
FINDINGS_NAME = STAGE0_NAME + ('-intruder-findings-v1' if CONDITIONAL_EXTENSION else ('-batch16-primary-findings-v3' if USE_BATCH16 else '-primary-findings-v1'))
FINDINGS_PLAN = ARTIFACT_ROOT / 'plans' / FINDINGS_NAME / 'stage0.findings.json'
run_stage0('findings-prepare', '--evaluation', EVALUATION_PLAN, '--drift', DRIFT_PLAN,
           '--name', FINDINGS_NAME)
FINDINGS_CONFIG = json.loads(FINDINGS_PLAN.read_text())
FINDINGS_DIR = ARTIFACT_ROOT / FINDINGS_CONFIG['run_path']
print('Registration:', FINDINGS_CONFIG['registration'])
print('Raw report:', FINDINGS_DIR / 'report.html')


## 18. Compute or resume paired results (CPU)
Builds the declared paired comparisons: accuracy/strict scores, median paired length, per-document KL, module write differences, and 10,000 paired bootstrap draws with seed 42. Saves raw tables, plots and HTML. Set a unit limit to stop after a few comparisons; rerunning skips sealed ones. No human labels are assumed.


In [ ]:
FINDINGS_MAX_NEW_UNITS = None
arguments = ['--config', FINDINGS_PLAN]
if FINDINGS_MAX_NEW_UNITS is not None:
    arguments += ['--max-new-units', FINDINGS_MAX_NEW_UNITS]
run_stage0('findings-run', *arguments)
run_stage0('findings-status', '--config', FINDINGS_PLAN)
print('Inspect:', FINDINGS_DIR / 'report.html')
print('Raw table exports:', FINDINGS_DIR / 'results')


## 19. Prepare the intended-answer review (CPU)
Targets changed extracted answers, invalid endpoints and last-number diagnostic disagreement. Strict-only format failures remain automatic tables. Identical prompt/response text is deduplicated. Optional reuse requires a verified historical audit configuration and explicit revision; no fuzzy text matching. The page hides source identity, gold and scorer results, but separate files are not an access-control boundary.


In [ ]:
REUSE_CONFIRMED_AUDIT = ''
REUSE_CONFIRMED_REVISION = ''
REUSE_ASSISTED_REPORT = ''
arguments = ['--config', FINDINGS_PLAN]
if REUSE_CONFIRMED_AUDIT.strip():
    arguments += ['--reuse-audit', REUSE_CONFIRMED_AUDIT]
    if REUSE_ASSISTED_REPORT.strip():
        arguments += ['--reuse-assisted-report', REUSE_ASSISTED_REPORT]
    if REUSE_CONFIRMED_REVISION.strip():
        arguments += ['--reuse-revision', REUSE_CONFIRMED_REVISION]
run_stage0('findings-audit-prepare', *arguments)
print('Open the reviewer page:', FINDINGS_DIR / 'audit' / 'reviewer' / 'review.html')
print('After saving labels, import the updated draft before continuing:',
      FINDINGS_DIR / 'audit' / 'reviewer' / 'draft.json')


## 20. Save an exported review draft (CPU; repeat as needed)
Enter the path to the downloaded draft. Numeric labels require a number, not a correctness decision. Saving merges with the durable revision; a stale draft is rejected. Incomplete review does not block a raw-results report.


In [ ]:
REVIEW_DRAFT_PATH = ''
if not REVIEW_DRAFT_PATH.strip():
    raise ValueError('Enter the path to the downloaded reviewer JSON draft.')
run_stage0('findings-audit-save', '--config', FINDINGS_PLAN, '--labels', REVIEW_DRAFT_PATH)
run_stage0('findings-audit-status', '--config', FINDINGS_PLAN)


## 21. Freeze and reveal completed review (CPU; only after all answers are labelled)
Freezes the identified answers before revealing gold/scorer comparisons. Caps keep their original zero-reward rule. Confirmed numeric cases form a separately labelled overlay; absent/ambiguous labels are retained without inventing a numeric correction. Corrections after freeze require a new named analysis/audit with exposed timing, preserving this one.


In [ ]:
FREEZE_COMPLETED_REVIEW = False
if FREEZE_COMPLETED_REVIEW:
    run_stage0('findings-audit-freeze', '--config', FINDINGS_PLAN)
    run_stage0('findings-audit-reveal', '--config', FINDINGS_PLAN)
else:
    run_stage0('findings-audit-status', '--config', FINDINGS_PLAN)


## 22. Verify and publish the report snapshot (CPU)
Reconstructs raw estimates and intervals against saved pairs and draws. Publishes a new immutable HTML/table snapshot with current review status; the original raw report remains intact. Pending review and absent optional experiments do not prevent publication. These files do not establish real execution when their source is a synthetic fixture.


In [ ]:
run_stage0('findings-verify', '--config', FINDINGS_PLAN)
run_stage0('findings-report', '--config', FINDINGS_PLAN)
print('Original raw report:', FINDINGS_DIR / 'report.html')
print('Review-status report snapshots:', FINDINGS_DIR / 'reports')
print('JSON / JSONL / CSV:', FINDINGS_DIR / 'results')
print('Exportable SVG / PNG figures:', FINDINGS_DIR / 'plots')


## 23. Prepare rotation and intruder geometry (optional CPU branch)
Ticket 08 measures both saved step-64 adapters against the verified base spectral cache. It does not require the primary GPU collections or report above: after CPU sections 1–8, you may jump here. All modules are measured, including small updates. No GPU/model forward passes or edits are run.


In [ ]:
SPECTRAL_NAME = STAGE0_NAME + '-spectral-v1'
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / SPECTRAL_NAME / 'stage0.spectral.json'
INTRUDER_NAME = STAGE0_NAME + '-intruder-geometry-v1'
INTRUDER_PLAN = ARTIFACT_ROOT / 'plans' / INTRUDER_NAME / 'stage0.intruders.json'
assert SPECTRAL_PLAN.is_file(), 'Restore and verify the base spectral cache first (sections 5–8).'
run_stage0('intruders-prepare', '--spectral', SPECTRAL_PLAN, '--name', INTRUDER_NAME)
INTRUDER_CONFIG = json.loads(INTRUDER_PLAN.read_text())
INTRUDER_DIR = ARTIFACT_ROOT / INTRUDER_CONFIG['run_path']
print('CPU SVD units:', len(INTRUDER_CONFIG['modules']) * len(INTRUDER_CONFIG['arms']))
print('Saved geometry:', INTRUDER_DIR)


## 24. Measure or resume updated geometry (CPU)
Start with one module/arm unit to measure actual SVD time and storage. Then set the unit limit to `None` to complete all missing units. There is no small-update skip. Each sealed unit retains singular values, enough left/right vectors, principal angles, gaps, top-40 overlaps and chance-aware eligibility. A stopped unit is recomputed; completed units are checked and reused.


In [ ]:
INTRUDER_MAX_NEW_UNITS = 1
arguments = ['--config', INTRUDER_PLAN]
if INTRUDER_MAX_NEW_UNITS is not None:
    arguments += ['--max-new-units', INTRUDER_MAX_NEW_UNITS]
run_stage0('intruders-run', *arguments)
run_stage0('intruders-status', '--config', INTRUDER_PLAN)
print('Per-unit timing and raw diagnostics:', INTRUDER_DIR / 'units')


## 25. Verify and inspect conditional eligibility (CPU)
Checks sealed payloads and reconstructs geometry diagnostics from retained bases, spectra and source factors. Inspect the HTML and raw tables. “Undecidable” is distinct from “no intruders”; ties also retain explicit limitations. This output is the input to the later conditional V6/V6c construction ticket, not permission to edit or run additional GPU jobs.


In [ ]:
run_stage0('intruders-verify', '--config', INTRUDER_PLAN)
geometry = json.loads((INTRUDER_DIR / 'results' / 'results.json').read_text())
print('Execution kind:', geometry['execution_kind'])
print('Eligibility by arm:', geometry['eligibility'])
print('Report:', INTRUDER_DIR / 'report.html')
print('All candidate values:', INTRUDER_DIR / 'results' / 'candidates.csv')
print('Rotation and gaps:', INTRUDER_DIR / 'results' / 'geometry.csv')


## 26. Prepare conditional intruder edits (optional CPU branch)
Requires the verified principal catalogue (sections 10–12) and updated geometry (23–25). Output: a frozen plan for full V6, matched V6/V6c and the ratio sensitivity. Controls use distinct non-flagged top-40 directions. An arm with no eligible flags remains explicitly not applicable. The storage figure is disk usage, not GPU memory.


In [ ]:
EDIT_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-principal-edits-v1') / 'stage0.edits.json'
INTRUDER_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-intruder-geometry-v1') / 'stage0.intruders.json'
CONDITIONAL_NAME = STAGE0_NAME + '-intruder-edits-v1'
CONDITIONAL_PLAN = ARTIFACT_ROOT / 'plans' / CONDITIONAL_NAME / 'stage0.intruder-edits.json'
assert EDIT_PLAN.is_file() and INTRUDER_PLAN.is_file(), 'Restore completed principal edits and geometry first.'
run_stage0('intruder-edits-prepare', '--edits', EDIT_PLAN, '--geometry', INTRUDER_PLAN, '--name', CONDITIONAL_NAME)
CONDITIONAL_CONFIG = json.loads(CONDITIONAL_PLAN.read_text())
CONDITIONAL_DIR = ARTIFACT_ROOT / CONDITIONAL_CONFIG['run_path']
print('Module/arm units:', len(CONDITIONAL_CONFIG['selections']))
print('Eligible module/arm units:', sum(bool(row['flagged_indices']) for row in CONDITIONAL_CONFIG['selections']))
print('Upper disk GiB:', CONDITIONAL_CONFIG['storage_upper_bound_bytes'] / 2**30)
print('Catalogue directory:', CONDITIONAL_DIR)


## 27. Build or resume merged matrices (CPU)
Start with one unit to measure time and storage; set the limit to `None` to finish only missing units. Full V6 remains separate from the matched comparison. The ratio sensitivity retains original pairs in [0.8, 1.25] without rematching. Source adapters and base weights are preserved.


In [ ]:
CONDITIONAL_MAX_NEW_UNITS = 1
arguments = ['--config', CONDITIONAL_PLAN]
if CONDITIONAL_MAX_NEW_UNITS is not None:
    arguments += ['--max-new-units', CONDITIONAL_MAX_NEW_UNITS]
run_stage0('intruder-edits-run', *arguments)
run_stage0('intruder-edits-status', '--config', CONDITIONAL_PLAN)
print('Saved matrices, construction checks and timing:', CONDITIONAL_DIR / 'units')


## 28. Verify conditional catalogue (CPU)
Reconstructs the intended operators from source weights, factors and saved singular components. Outputs full matrices, exact adapter-disable masks, matched-pair ratios and exclusion reasons. No eligible flags produces a verified not-applicable receipt. Ticket 10 separately qualifies the unhalved mixed loader before any edited-model evaluation; this stage does not authorize GPU execution.


In [ ]:
run_stage0('intruder-edits-verify', '--config', CONDITIONAL_PLAN)
conditional = json.loads((CONDITIONAL_DIR / 'results' / 'catalogue.json').read_text())
print('State:', conditional['state'])
print('Applicability:', {arm: row['status'] for arm, row in conditional['arms'].items()})
for name, variant in conditional['variants'].items():
    print(name, '| modules:', variant['modules_edited'], '|', variant['status'])
print('Report:', CONDITIONAL_DIR / 'report.html')
print('Matrices and masks:', CONDITIONAL_DIR / 'results' / 'catalogue.json')
print('Matched pairs:', CONDITIONAL_DIR / 'results' / 'matching.csv')
print('GPU qualified:', conditional['gpu_qualified'])


## 29. Audit the optional R0 sampling source (CPU)
Requires the completed source audit (sections 1–4), not the intervention or GPU cells. Restore the original baseline configuration, raw batch files and runtime metadata. Output: verified old groups and an explicit missing-file list. An aggregate report is insufficient. R0 remains separate from the primary/conditional findings.


In [ ]:
R0_NAME = STAGE0_NAME + '-untuned-rollouts-v1'
R0_BASELINE = ARTIFACT_ROOT / 'plans' / 'pilot4-untuned-sampling-v1' / 'baseline.config.json'
run_stage0('rollouts-audit', '--audit', PLAN, '--baseline', R0_BASELINE, '--name', R0_NAME)
R0_DIR = ARTIFACT_ROOT / 'runs' / 'stage0' / R0_NAME / 'rollouts'
print('Source audit:', R0_DIR / 'inputs' / 'source-audit.json')


## 30. Prepare the optional R0 plan (CPU)
Verifies the pinned local weights and freezes training inputs, old raw draws, source hashes and explicit seeds. No generation runs here. Two exact historical eight-draw replays may permit old128 + new384; otherwise the GPU session produces a fresh uniform512 cohort. The original128 remain archived. Missing seeds/order cannot silently select a join. Parquet support must be installed before GPU work.


In [ ]:
import importlib.util
INSTALL_R0_TABLE_DEPENDENCY = False
if importlib.util.find_spec('pyarrow') is None:
    if not INSTALL_R0_TABLE_DEPENDENCY:
        raise RuntimeError('Parquet support is missing. Set INSTALL_R0_TABLE_DEPENDENCY=True and rerun on CPU.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyarrow>=14'], check=True)
source_audit = json.loads((ARTIFACT_ROOT / 'runs' / STAGE0_NAME / 'results' / 'source-audit.json').read_text())
WEIGHTS_DIR = ARTIFACT_ROOT / 'inputs' / 'models' / source_audit['sources']['sft']['identity']['model_revision']
R0_PLAN = ARTIFACT_ROOT / 'plans' / R0_NAME / 'stage0.rollouts.json'
run_stage0('rollouts-prepare', '--audit', PLAN, '--baseline', R0_BASELINE,
           '--weights-dir', WEIGHTS_DIR, '--name', R0_NAME)
print('Frozen plan:', R0_PLAN)
print('Next: GPU setup 1–4, then optional R0 sections 20–25. Keep the cumulative 20-unit ledger.')


## 31. Verify and inspect R0 outputs (CPU)
Run after R0 GPU collection and usage reconciliation. Reconstructs all 512×8 identities, stop/scoring fields, cohort memberships and the Parquet table from sealed groups. Outputs here are sampling/cohort evidence; NLL, entropy and gradients belong to later supporting stages.


In [ ]:
R0_NAME = STAGE0_NAME + '-untuned-rollouts-v1'
R0_PLAN = ARTIFACT_ROOT / 'plans' / R0_NAME / 'stage0.rollouts.json'
run_stage0('rollouts-verify', '--config', R0_PLAN)
R0_DIR = ARTIFACT_ROOT / json.loads(R0_PLAN.read_text())['run_path']
result = json.loads((R0_DIR / 'results' / 'results.json').read_text())
print('Route:', result['route'])
print('Cohort counts:', result['counts'])
print('Raw JSONL:', R0_DIR / 'results' / 'rollouts.jsonl')
print('Raw Parquet:', R0_DIR / 'results' / 'rollouts.parquet')
print('Memberships:', R0_DIR / 'results' / 'datasets.json')
print('Report:', R0_DIR / 'report.html')


## 32. Prepare optional answer NLL measurement (CPU)
Consumes verified R0 results and deduplicates exact prompt/answer token sequences across all four datasets. Gold supervision follows saved SFT labels through EOS. Sampled answers use observed generated IDs, including EOS when present and no synthetic EOS on cap. No GPU scoring runs here. Keep the R0 sampled runtime available for the later scoring pass.


In [ ]:
R0_NAME = STAGE0_NAME + '-untuned-rollouts-v1'
R0_PLAN = ARTIFACT_ROOT / 'plans' / R0_NAME / 'stage0.rollouts.json'
DISTANCE_NAME = STAGE0_NAME + '-answer-distance-v1'
DISTANCE_PLAN = ARTIFACT_ROOT / 'plans' / DISTANCE_NAME / 'stage0.distance.json'
run_stage0('distance-prepare', '--rollouts', R0_PLAN, '--name', DISTANCE_NAME,
           '--context-chunk', 16, '--workspace-bytes', 512 * 1024 * 1024)
config = json.loads(DISTANCE_PLAN.read_text())
print('Unique answers:', len(config['unique_answer_ids']))
print('Frozen scoring plan:', DISTANCE_PLAN)
print('Next: GPU sections 26–31 in the sampled production runtime.')


## 33. Aggregate answer scores and verify report (CPU)
Run after GPU scoring/reconciliation. Reconstructs token-weighted means and whole-problem bootstrap intervals. NLL is primary; excess surprise is descriptive, not KL. The ±0.02 own-all gate is retained; failure preserves evidence and requires review. Whole-answer −log(k/8) is a separate field and receives no per-token agreement test.

The report also includes answer-weighted sensitivity means/intervals, short-answer counts and all three window token counts. Whole problems remain the bootstrap units.


In [ ]:
DISTANCE_NAME = STAGE0_NAME + '-answer-distance-v1'
DISTANCE_PLAN = ARTIFACT_ROOT / 'plans' / DISTANCE_NAME / 'stage0.distance.json'
run_stage0('distance-report', '--config', DISTANCE_PLAN)
run_stage0('distance-verify', '--config', DISTANCE_PLAN)
DISTANCE_DIR = ARTIFACT_ROOT / json.loads(DISTANCE_PLAN.read_text())['run_path']
print('Report:', DISTANCE_DIR / 'report.html')
print('Summary:', DISTANCE_DIR / 'summary.parquet')
print('Per-token evidence:', DISTANCE_DIR / 'units')
print('Per-problem and top-gold-token tables:', DISTANCE_DIR / 'results')


## 34. Prepare optional initial-gradient comparison (CPU)
Requires the verified spectral cache and completed R0 cohorts. Hashes both saved A₀ payloads: three objectives at one A₀, or three at each of two distinct A₀s. Gold and own-correct share eligible question batches; GRPO retains all512. No model call or optimizer step. GPU sections32–37 collect gradients in their own frozen session; saved R0 tokens do not need regenerating.


In [ ]:
PUSH_NAME = STAGE0_NAME + '-initial-pushes-v1'
PUSH_PLAN = ARTIFACT_ROOT / 'plans' / PUSH_NAME / 'stage0.pushes.json'
R0_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-untuned-rollouts-v1') / 'stage0.rollouts.json'
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-spectral-v1') / 'stage0.spectral.json'
assert SPECTRAL_PLAN.is_file(), 'Select the existing verified spectral plan from CPU section5.'
run_stage0('pushes-prepare', '--spectral', SPECTRAL_PLAN, '--rollouts', R0_PLAN,
           '--name', PUSH_NAME, '--context-chunk', 16)
PUSH_CONFIG = json.loads(PUSH_PLAN.read_text())
print('A0 audit:', PUSH_CONFIG['initialization_audit'])
print('Cohorts:', PUSH_CONFIG['cohorts'])
print('Gradient batches:', len(PUSH_CONFIG['unit_keys']))
print('Next: GPU setup1–4, then32–37.')

## 35. Summarize initial pushes (CPU)
After GPU collection/reconciliation. Saves batch-mean pushes with/without the final partial batch, token-weighted means, consistency excluding partial batches, principal output overlap and same-A₀ direction comparisons. No endpoint alignments are revealed here. Raw norms are descriptive within each condition.


In [ ]:
PUSH_NAME = STAGE0_NAME + '-initial-pushes-v1'
PUSH_PLAN = ARTIFACT_ROOT / 'plans' / PUSH_NAME / 'stage0.pushes.json'
run_stage0('pushes-verify-measurement', '--config', PUSH_PLAN)
run_stage0('pushes-report', '--config', PUSH_PLAN)
PUSH_DIR = ARTIFACT_ROOT / json.loads(PUSH_PLAN.read_text())['run_path']
print('Report:', PUSH_DIR / 'report.html')
print('Summary:', PUSH_DIR / 'results' / 'summary.json')
print('Mean vectors:', PUSH_DIR / 'results' / 'mean-pushes.npz')
print('Raw per-batch gradients:', PUSH_DIR / 'units')

## 36. Freeze conditional predictions, then reveal endpoints (CPU)
Write predictions after inspecting section35, before endpoint/previous trained-overlap reveal. Declare any prior exposure honestly; existing exposure receipts force exploratory status. The reveal orients paired (a64,b64) by a64·A₀, reports flips/undefined signs, and compares pushes with endpoint b and Gaussian controls. This is supporting alignment evidence, not an AdamW reconstruction or causal result.


In [ ]:
PUSH_PREDICTIONS = ''
PUSH_PRIOR_EXPOSURE = None  # Set True or False explicitly.
PUSH_PREDICTION_NOTES = ''
if (not PUSH_PREDICTIONS.strip() or type(PUSH_PRIOR_EXPOSURE) is not bool
        or not PUSH_PREDICTION_NOTES.strip()):
    raise ValueError('Enter conditional predictions, prior exposure and review notes before reveal.')
review = dict(reviewer='Prasad', notes=PUSH_PREDICTION_NOTES,
              predictions=PUSH_PREDICTIONS, prior_exposure=PUSH_PRIOR_EXPOSURE)
review_path = ARTIFACT_ROOT / 'notebook' / PUSH_NAME / 'prediction-review.json'
review_path.parent.mkdir(parents=True, exist_ok=True)
review_path.write_text(json.dumps(review, indent=2) + '\n')
run_stage0('pushes-predict', '--config', PUSH_PLAN, '--review', review_path)
run_stage0('pushes-reveal', '--config', PUSH_PLAN)
run_stage0('pushes-verify', '--config', PUSH_PLAN)
print('Endpoint alignments:', PUSH_DIR / 'results' / 'endpoints.json')

## 37. Prepare optional base-input energy measurement (CPU)
Requires the verified spectral cache, reviewed Ticket04 qualification and original fixed-input product. Prepares SFT/GRPO step0 and64 plus the saved random64 control, at k30/k50/k70. Independently checks normalized projections against Ticket02. No model call. One later base forward per document supplies every condition.


In [ ]:
ENERGY_NAME = STAGE0_NAME + '-input-energy-v1'
ENERGY_PLAN = ARTIFACT_ROOT / 'plans' / ENERGY_NAME / 'stage0.energy.json'
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-spectral-v1') / 'stage0.spectral.json'
QUALIFICATION_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-loader-qualification-v1') / 'stage0.qualification.json'
audit_path = ARTIFACT_ROOT / 'runs' / STAGE0_NAME / 'results' / 'source-audit.json'
audit = json.loads(audit_path.read_text())
ENERGY_INPUTS = ARTIFACT_ROOT / audit['sources']['sft']['path']
RANDOM_SOURCE = ARTIFACT_ROOT / (audit['sources'].get('random', {}).get('path')
    or 'plans/pilot4-grpo-fp32-v1-random-writes-v1/activation.prepared.json')
run_stage0('energy-prepare', '--spectral', SPECTRAL_PLAN, '--qualification', QUALIFICATION_PLAN,
           '--inputs', ENERGY_INPUTS, '--random', RANDOM_SOURCE, '--name', ENERGY_NAME,
           '--batches-per-unit', 8)
ENERGY_CONFIG = json.loads(ENERGY_PLAN.read_text())
ENERGY_DIR = ARTIFACT_ROOT / ENERGY_CONFIG['run_path']
print('Factors:', ENERGY_CONFIG['factor_labels'])
print('Cutoffs:', ENERGY_CONFIG['energy_levels'])
print('Base forwards: 300 total; save groups:', len(ENERGY_CONFIG['units']))
print('Projection checks:', ENERGY_DIR / 'checks' / 'projections.json')
print('Next: freeze predictions in38 before GPU profile.')

## 38. Freeze energy hypotheses before measurement (CPU)
Declare predictions and prior inspection honestly. Existing Ticket02 exposure or declared inspection marks interpretation exploratory. This does not block the primary intervention workflow. The GPU profile and production share this declaration.


In [ ]:
ENERGY_PREDICTIONS = ''
ENERGY_PRIOR_EXPOSURE = None  # Set True or False explicitly.
ENERGY_PREDICTION_NOTES = ''
if (not ENERGY_PREDICTIONS.strip() or type(ENERGY_PRIOR_EXPOSURE) is not bool
        or not ENERGY_PREDICTION_NOTES.strip()):
    raise ValueError('Enter energy hypotheses, prior exposure and review notes.')
review = dict(reviewer='Prasad', predictions=ENERGY_PREDICTIONS,
              prior_exposure=ENERGY_PRIOR_EXPOSURE, notes=ENERGY_PREDICTION_NOTES)
review_path = ARTIFACT_ROOT / 'notebook' / ENERGY_NAME / 'prediction-review.json'
review_path.parent.mkdir(parents=True, exist_ok=True)
review_path.write_text(json.dumps(review, indent=2) + '\n')
run_stage0('energy-predict', '--config', ENERGY_PLAN, '--review', review_path)
print('Next: GPU setup1–4, then38–43.')

## 39. Reconstruct energy summaries and plots (CPU)
After collection/reconciliation, reconstruct pooled-token energies, the bounded principal/(principal+rest) headline and all three total-normalized components. Negative cross means cancellation; positive means reinforcement. Zero denominators stay undefined. Save module tables, layer/type summaries, geometric comparisons, plots and provenance. No GPU is needed.


In [ ]:
ENERGY_NAME = STAGE0_NAME + '-input-energy-v1'
ENERGY_PLAN = ARTIFACT_ROOT / 'plans' / ENERGY_NAME / 'stage0.energy.json'
run_stage0('energy-verify-measurement', '--config', ENERGY_PLAN)
run_stage0('energy-report', '--config', ENERGY_PLAN)
run_stage0('energy-verify', '--config', ENERGY_PLAN)
ENERGY_DIR = ARTIFACT_ROOT / json.loads(ENERGY_PLAN.read_text())['run_path']
print('Report:', ENERGY_DIR / 'report.html')
print('Raw sufficient sums:', ENERGY_DIR / 'units')
print('Module and layer/type tables:', ENERGY_DIR / 'results')
print('Plots:', ENERGY_DIR / 'plots')

## 40. Audit optional checkpoint-trajectory sources (CPU)
Select existing saved-workflow configs, not HTML reports. Original steps0/8/16/32/64 are audited; missing checkpoints/products are recorded rather than fabricated. Accepted product kinds: stage0-drift, pilot4-tokens, pilot3-writes, trajectory-collection. Saved endpoint lengths may use a verified stage0-evaluation or behaviour config. No model inference or new generation.


In [ ]:
TRAJECTORY_NAME = STAGE0_NAME + '-checkpoint-trajectories-v1'
TRAJECTORY_PLAN = ARTIFACT_ROOT / 'plans' / TRAJECTORY_NAME / 'stage0.trajectory.json'
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-spectral-v1') / 'stage0.spectral.json'
audit = json.loads((ARTIFACT_ROOT / 'runs' / STAGE0_NAME / 'results' / 'source-audit.json').read_text())
TRAJECTORY_INPUTS = ARTIFACT_ROOT / audit['sources']['sft']['path']
SAVED_TRAJECTORY_PRODUCTS = []
# Example: {'kind': 'stage0-drift', 'config': 'plans/<name>/stage0.drift.json'}
# Older products: {'kind': 'pilot4-tokens', 'config': 'plans/<name>/tokens.execution.json'}
SAVED_ENDPOINT_OUTPUTS = None
# Example: {'kind': 'behaviour', 'config': 'plans/<name>/behaviour.config.json'}
selection = dict(products=SAVED_TRAJECTORY_PRODUCTS,
                 random=(audit['sources'].get('random', {}).get('path')
                         or 'plans/pilot4-grpo-fp32-v1-random-writes-v1/activation.prepared.json'))
if SAVED_ENDPOINT_OUTPUTS is not None:
    selection['evaluation'] = SAVED_ENDPOINT_OUTPUTS
selection_path = ARTIFACT_ROOT / 'notebook' / TRAJECTORY_NAME / 'selection.json'
selection_path.parent.mkdir(parents=True, exist_ok=True)
if selection_path.exists() and json.loads(selection_path.read_text()) != selection:
    raise ValueError('This name already selects different sources; retain it and choose a new TRAJECTORY_NAME.')
selection_path.write_text(json.dumps(selection, indent=2) + '\n')
run_stage0('trajectory-prepare', '--spectral', SPECTRAL_PLAN, '--inputs', TRAJECTORY_INPUTS,
           '--selection', selection_path, '--name', TRAJECTORY_NAME)
run_stage0('trajectory-status', '--config', TRAJECTORY_PLAN)

## 41. Compute geometry and import saved trajectories (CPU)
Computes principal input/output overlap at k30/k50/k70 and verifies selected raw saved products. Resume skips sealed CPU units. Unknown or different sessions remain separate, even on the same GPU type. Historical raw-coefficient sensitivity stays descriptive; its strict repeat/chunk/KL gates are not changed.


In [ ]:
TRAJECTORY_NAME = STAGE0_NAME + '-checkpoint-trajectories-v1'
TRAJECTORY_PLAN = ARTIFACT_ROOT / 'plans' / TRAJECTORY_NAME / 'stage0.trajectory.json'
TRAJECTORY_MAX_CPU_UNITS = None
arguments = ['--config', TRAJECTORY_PLAN]
if TRAJECTORY_MAX_CPU_UNITS is not None:
    arguments += ['--max-new-units', TRAJECTORY_MAX_CPU_UNITS]
run_stage0('trajectory-run', *arguments)
run_stage0('trajectory-status', '--config', TRAJECTORY_PLAN)
print('CPU reporting can run now. GPU44–49 are optional, for missing measurements only.')

## 42. Publish trajectory snapshot and saved-output lengths (CPU)
Primary size is total Frobenius sqrt(sum module norm-product squared). FineWeb token-mean KL is primary; equal-document means and whole-document bootstrap intervals remain alongside it. Effective write is the equal-module mean of per-module token-mean ratios; all module values are retained. Log–log plots keep step0 in a separate linear-origin inset. Slopes and KL/size² are descriptive; “near2” is unclassified. Incomplete optional coverage is explicit. Choose a new report name when adding later collections.


In [ ]:
TRAJECTORY_REPORT_NAME = TRAJECTORY_NAME + '-report-v1'
TRAJECTORY_COLLECTION_NAME = TRAJECTORY_NAME + '-collection-v1'
TRAJECTORY_COLLECTION_PLAN = ARTIFACT_ROOT / 'plans' / TRAJECTORY_COLLECTION_NAME / 'stage0.trajectory.collection.json'
INCLUDE_TRAJECTORY_COLLECTION = False
arguments = ['--config', TRAJECTORY_PLAN, '--name', TRAJECTORY_REPORT_NAME]
if INCLUDE_TRAJECTORY_COLLECTION:
    arguments += ['--collection', TRAJECTORY_COLLECTION_PLAN]
run_stage0('trajectory-report', *arguments)
run_stage0('trajectory-verify', '--config', TRAJECTORY_PLAN, '--name', TRAJECTORY_REPORT_NAME)
print('Report:', ARTIFACT_ROOT / 'reports' / TRAJECTORY_REPORT_NAME / 'report.html')
print('Raw tables, bootstrap indices and slopes:', ARTIFACT_ROOT / 'reports' / TRAJECTORY_REPORT_NAME / 'results')
print('Exact equal-size SFT64 rescaling is a proposed follow-up only; it is not run here.')